# Лабораторна робота №2. Робота з SQL-базами: створення, запити, об'єднання таблиць

**Мета роботи:** опанувати основи роботи з реляційними базами даних через SQL: створення таблиць, виконання запитів, фільтрація, сортування та об'єднання таблиць.

## Крок 1. Підключення до бази даних і створення таблиць

In [1]:
import sqlite3

# Підключення до бази (файл створюється, якщо його ще немає)
conn = sqlite3.connect("university.db")
cursor = conn.cursor()

# Вмикаємо перевірку зовнішніх ключів (у SQLite вона вимкнена за замовчуванням)
cursor.execute("PRAGMA foreign_keys = ON")

# Щоб ноутбук можна було перезапускати, спершу видаляємо старі таблиці
cursor.executescript("""
DROP TABLE IF EXISTS grades;
DROP TABLE IF EXISTS students;
DROP TABLE IF EXISTS courses;
""")

# Таблиця студентів
cursor.execute("""
CREATE TABLE students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER,
    group_name TEXT
)
""")

# Таблиця оцінок із зовнішнім ключем на students
cursor.execute("""
CREATE TABLE grades (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    subject TEXT,
    grade INTEGER,
    FOREIGN KEY (student_id) REFERENCES students(id)
)
""")

conn.commit()
print("Таблиці створено успішно.")

Таблиці створено успішно.


## Крок 2. Додавання даних

In [2]:
students_data = [
    ("Іван Петренко", 20, "КН-21"),
    ("Марія Коваленко", 19, "КН-22"),
    ("Петро Шевченко", 21, "КН-21"),
    ("Олена Бондаренко", 20, "КН-22"),
]
grades_data = [
    (1, "Математика", 90),
    (1, "Програмування", 85),
    (2, "Математика", 92),
    (3, "Програмування", 78),
    (4, "Математика", 88),
    (4, "Програмування", 91),
]

cursor.executemany(
    "INSERT INTO students (name, age, group_name) VALUES (?, ?, ?)", students_data
)
cursor.executemany(
    "INSERT INTO grades (student_id, subject, grade) VALUES (?, ?, ?)", grades_data
)
conn.commit()
print("Дані додано.")

Дані додано.


## Крок 3. SQL-запити

**3.1. Усі студенти**

In [3]:
cursor.execute("SELECT * FROM students")
for row in cursor.fetchall():
    print(row)

(1, 'Іван Петренко', 20, 'КН-21')
(2, 'Марія Коваленко', 19, 'КН-22')
(3, 'Петро Шевченко', 21, 'КН-21')
(4, 'Олена Бондаренко', 20, 'КН-22')


**3.2. Студенти старші за 19 років**

In [4]:
cursor.execute("SELECT name, age FROM students WHERE age > 19")
for row in cursor.fetchall():
    print(row)

('Іван Петренко', 20)
('Петро Шевченко', 21)
('Олена Бондаренко', 20)


**3.3. Оцінки з «Програмування» (JOIN)**

In [5]:
cursor.execute("""
SELECT s.name, g.grade
FROM students s
JOIN grades g ON s.id = g.student_id
WHERE g.subject = 'Програмування'
""")
for row in cursor.fetchall():
    print(row)

('Іван Петренко', 85)
('Петро Шевченко', 78)
('Олена Бондаренко', 91)


**3.4. Сортування:** за віком за спаданням, за однакового віку — за іменем

In [6]:
cursor.execute("SELECT name, age FROM students ORDER BY age DESC, name")
for row in cursor.fetchall():
    print(row)

('Петро Шевченко', 21)
('Іван Петренко', 20)
('Олена Бондаренко', 20)
('Марія Коваленко', 19)


## Крок 4. Агрегатні функції

In [7]:
cursor.execute("SELECT subject, AVG(grade) FROM grades GROUP BY subject")
for row in cursor.fetchall():
    print(row)

('Математика', 90.0)
('Програмування', 84.66666666666667)


Той самий результат у вигляді таблиці через `pandas`:

In [8]:
import pandas as pd

pd.read_sql_query(
    "SELECT subject AS \"Предмет\", AVG(grade) AS \"Середня оцінка\" FROM grades GROUP BY subject",
    conn,
)

,Предмет,Середня оцінка
0,Математика,90.000000
1,Програмування,84.666667


## Самостійне завдання

### 1–2. Таблиця `courses` і додавання курсів

Курс «Бази даних» навмисно не має оцінок — це покаже різницю між `INNER JOIN` та `LEFT JOIN`.

In [9]:
cursor.execute("""
CREATE TABLE courses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL,
    hours INTEGER
)
""")
courses_data = [("Математика", 120), ("Програмування", 150), ("Бази даних", 90)]
cursor.executemany("INSERT INTO courses (name, hours) VALUES (?, ?)", courses_data)
conn.commit()
pd.read_sql_query("SELECT * FROM courses", conn)

,id,name,hours
0,1,Математика,120
1,2,Програмування,150
2,3,Бази даних,90


### 3. Звіт: `Ім'я | Курс | Оцінка | Години`

In [10]:
report_sql = """
SELECT s.name AS "Ім'я", c.name AS "Курс", g.grade AS "Оцінка", c.hours AS "Години"
FROM students s
JOIN grades g ON s.id = g.student_id
JOIN courses c ON g.subject = c.name
ORDER BY s.name, c.name
"""
cursor.execute(report_sql)
report_header = [d[0] for d in cursor.description]
report_rows = cursor.fetchall()

print(f"{report_header[0]:<18} | {report_header[1]:<14} | {report_header[2]:<6} | {report_header[3]}")
print("-" * 55)
for name, course, grade, hours in report_rows:
    print(f"{name:<18} | {course:<14} | {grade:<6} | {hours}")

Ім'я               | Курс           | Оцінка | Години
-------------------------------------------------------
Іван Петренко      | Математика     | 90     | 120
Іван Петренко      | Програмування  | 85     | 150
Марія Коваленко    | Математика     | 92     | 120
Олена Бондаренко   | Математика     | 88     | 120
Олена Бондаренко   | Програмування  | 91     | 150
Петро Шевченко     | Програмування  | 78     | 150


In [11]:
pd.read_sql_query(report_sql, conn)

,Ім'я,Курс,Оцінка,Години
0,Іван Петренко,Математика,90,120
1,Іван Петренко,Програмування,85,150
2,Марія Коваленко,Математика,92,120
3,Олена Бондаренко,Математика,88,120
4,Олена Бондаренко,Програмування,91,150
5,Петро Шевченко,Програмування,78,150


### 4. Середня оцінка по кожному курсу

Використано `LEFT JOIN` від `courses`, тому курс «Бази даних» без оцінок не зникає з результату: його середня — `NULL`, кількість оцінок — 0. `INNER JOIN` такий курс би відкинув.

In [12]:
avg_sql = """
SELECT c.name AS course,
       ROUND(AVG(g.grade), 2) AS avg_grade,
       COUNT(g.id) AS grades_count
FROM courses c
LEFT JOIN grades g ON g.subject = c.name
GROUP BY c.id
ORDER BY c.name
"""
cursor.execute(avg_sql)
avg_header = [d[0] for d in cursor.description]
avg_rows = cursor.fetchall()
pd.read_sql_query(avg_sql, conn)

,course,avg_grade,grades_count
0,Бази даних,NaN,0
1,Математика,90.00,3
2,Програмування,84.67,3


### 5. Збереження результатів у CSV

Кодування `utf-8-sig` потрібне, щоб Excel коректно відкривав кирилицю.

In [13]:
import csv

with open("report.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(report_header)
    writer.writerows(report_rows)

with open("course_avg.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(avg_header)
    writer.writerows(avg_rows)

for fname in ("report.csv", "course_avg.csv"):
    print("==", fname)
    print(open(fname, encoding="utf-8-sig").read())

== report.csv
Ім'я,Курс,Оцінка,Години
Іван Петренко,Математика,90,120
Іван Петренко,Програмування,85,150
Марія Коваленко,Математика,92,120
Олена Бондаренко,Математика,88,120
Олена Бондаренко,Програмування,91,150
Петро Шевченко,Програмування,78,150

== course_avg.csv
course,avg_grade,grades_count
Бази даних,,0
Математика,90.0,3
Програмування,84.67,3



## Крок 5. Закриття з'єднання

In [14]:
conn.close()
print("З'єднання закрито.")

З'єднання закрито.


## Контрольні питання

**1. Що таке реляційна база даних?**
Реляційна база даних — це база, у якій дані зберігаються у вигляді таблиць (відношень). Рядок таблиці — це запис, стовпець — атрибут певного типу. Кожен запис однозначно визначається первинним ключем (`PRIMARY KEY`), а таблиці пов'язуються між собою через зовнішні ключі (`FOREIGN KEY`). Для роботи з даними використовують мову SQL. Цілісність даних забезпечують обмеження (`NOT NULL`, `UNIQUE`, `CHECK`, ключі) і транзакції з властивостями ACID. Приклади СУБД: SQLite, PostgreSQL, MySQL, SQL Server.

**2. Які основні типи команд SQL ви знаєте?**
- **DDL** (Data Definition Language) — визначення структури: `CREATE`, `ALTER`, `DROP`, `TRUNCATE`.
- **DML** (Data Manipulation Language) — робота з даними: `SELECT`, `INSERT`, `UPDATE`, `DELETE`. `SELECT` іноді виділяють в окрему групу DQL (Data Query Language).
- **DCL** (Data Control Language) — права доступу: `GRANT`, `REVOKE`.
- **TCL** (Transaction Control Language) — керування транзакціями: `COMMIT`, `ROLLBACK`, `SAVEPOINT`.

**3. Як створити зв'язок між таблицями?**
У дочірній таблиці додають стовпець, який посилається на первинний ключ батьківської таблиці, і оголошують його зовнішнім ключем:
```sql
CREATE TABLE grades (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    subject TEXT,
    grade INTEGER,
    FOREIGN KEY (student_id) REFERENCES students(id)
);
```
Так створюється зв'язок «один-до-багатьох»: в одного студента може бути багато оцінок. Зв'язок «багато-до-багатьох» реалізують через проміжну таблицю з двома зовнішніми ключами. Пов'язані дані з кількох таблиць отримують запитом із `JOIN ... ON`. У SQLite перевірку зовнішніх ключів треба ввімкнути окремо: `PRAGMA foreign_keys = ON`.

**4. Для чого використовуються агрегатні функції?**
Агрегатні функції обчислюють одне значення з набору рядків: `COUNT` (кількість), `SUM` (сума), `AVG` (середнє), `MIN`, `MAX`. Разом із `GROUP BY` вони рахують показники для кожної групи окремо, наприклад середній бал по кожному предмету. `HAVING` фільтрує вже згруповані результати:
```sql
SELECT subject, AVG(grade) FROM grades GROUP BY subject HAVING AVG(grade) > 85;
```
Їх використовують для звітів, статистики та зведення великих обсягів даних.

**5. Чим відрізняються INNER JOIN, LEFT JOIN, RIGHT JOIN?**
- `INNER JOIN` повертає лише ті рядки, для яких знайшлася пара в обох таблицях.
- `LEFT JOIN` повертає всі рядки лівої таблиці. Якщо пари в правій таблиці немає, її стовпці заповнюються `NULL`. Наприклад, курс без жодної оцінки все одно потрапить у результат.
- `RIGHT JOIN` працює дзеркально: повертає всі рядки правої таблиці, а відсутні дані лівої заповнює `NULL`. SQLite підтримує його з версії 3.39.
- Є також `FULL OUTER JOIN`: він повертає всі рядки з обох таблиць.

## Висновки

У ході роботи створено базу SQLite `university.db` з таблицями `students`, `grades` і `courses`, пов'язаними зовнішнім ключем, та заповнено їх за допомогою `executemany`. Виконано запити з фільтрацією (`WHERE`), сортуванням (`ORDER BY`), об'єднанням трьох таблиць (`JOIN`) і агрегуванням (`AVG`, `GROUP BY`). На прикладі курсу без оцінок показано різницю між `INNER JOIN` і `LEFT JOIN`. Результати збережено у файли `report.csv` та `course_avg.csv` засобами Python.